In [1]:
import { display } from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf8");
display.html(`<style>${css}</style>`);

In [2]:
const { execSync } = await import('child_process');
console.log(execSync('npm install chevrotain@10').toString());
console.log(execSync('npm install readline-sync').toString());
import { createToken, Lexer,  CstParser } from "chevrotain";


up to date, audited 16 packages in 809ms

1 package is looking for funding
  run `npm fund` for details

found 0 vulnerabilities




up to date, audited 16 packages in 823ms

1 package is looking for funding
  run `npm fund` for details

found 0 vulnerabilities



In [3]:
import readlineSync from "readline-sync";

# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using `Ply`.  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]   
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}                               
\end{array}
$$

## Specification of the Scanner

The next line is necessary because we use `Ply` from a notebook.

We generate the lexer.

In [4]:
// Lexer Definition (entspricht t_NUMBER, t_IDENTIFIER, etc.)

function createCalcLexer() {
  const NumberTok = createToken({
    name: "NUMBER",
    pattern: /(0|[1-9][0-9]*)(\.[0-9]*)?/,
  });
  const Identifier = createToken({
    name: "IDENTIFIER",
    pattern: /[a-zA-Z][a-zA-Z0-9_]*/,
  });
  const Assign = createToken({ name: "ASSIGN", pattern: /:=/ });
  const Plus = createToken({ name: "PLUS", pattern: /\+/ });
  const Minus = createToken({ name: "MINUS", pattern: /-/ });
  const Mul = createToken({ name: "MUL", pattern: /\*/ });
  const Div = createToken({ name: "DIV", pattern: /\// });
  const LParen = createToken({ name: "LPAREN", pattern: /\(/ });
  const RParen = createToken({ name: "RPAREN", pattern: /\)/ });
  const Semicolon = createToken({ name: "SEMICOLON", pattern: /;/ });
  const WhiteSpace = createToken({
    name: "WhiteSpace",
    pattern: /[ \t\r]+/,
    group: Lexer.SKIPPED,
  });

  const allTokens = [
    WhiteSpace,
    Assign,
    Plus,
    Minus,
    Mul,
    Div,
    LParen,
    RParen,
    Semicolon,
    NumberTok,
    Identifier,
  ];

  return {
    allTokens,
    tokens: { NumberTok, Identifier, Assign, Plus, Minus, Mul, Div, LParen, RParen, Semicolon },
    lexer: new Lexer(allTokens),
  };
}

const { allTokens, tokens, lexer: CalcLexer } = createCalcLexer();

Unkown characters are reported as lexical errors.

## Specification of the Parser

In [5]:
// Lexer Definition (entspricht t_NUMBER, t_IDENTIFIER, etc.)

function createCalcLexer() {
  const NumberTok = createToken({
    name: "NUMBER",
    pattern: /(0|[1-9][0-9]*)(\.[0-9]*)?/,
  });
  const Identifier = createToken({
    name: "IDENTIFIER",
    pattern: /[a-zA-Z][a-zA-Z0-9_]*/,
  });
  const Assign = createToken({ name: "ASSIGN", pattern: /:=/ });
  const Plus = createToken({ name: "PLUS", pattern: /\+/ });
  const Minus = createToken({ name: "MINUS", pattern: /-/ });
  const Mul = createToken({ name: "MUL", pattern: /\*/ });
  const Div = createToken({ name: "DIV", pattern: /\// });
  const LParen = createToken({ name: "LPAREN", pattern: /\(/ });
  const RParen = createToken({ name: "RPAREN", pattern: /\)/ });
  const Semicolon = createToken({ name: "SEMICOLON", pattern: /;/ });
  const WhiteSpace = createToken({
    name: "WhiteSpace",
    pattern: /[ \t\r]+/,
    group: Lexer.SKIPPED,
  });

  const allTokens = [
    WhiteSpace,
    Assign,
    Plus,
    Minus,
    Mul,
    Div,
    LParen,
    RParen,
    Semicolon,
    NumberTok,
    Identifier,
  ];

  return {
    allTokens,
    tokens: { NumberTok, Identifier, Assign, Plus, Minus, Mul, Div, LParen, RParen, Semicolon },
    lexer: new Lexer(allTokens),
  };
}

const { allTokens, tokens, lexer: CalcLexer } = createCalcLexer();

In [7]:
// ======================
// 3️⃣  PARSER-DEFINITION
// ======================
// Diese Zelle definiert den Parser mit separaten Regel-Funktionen.
// Es gibt KEINE exportierte Klasse → keine protected/private Fehler.
// ---------------------------------------------------------------------------

function createCalcParser(allTokens: any, tokens: any) {
  // Lokale Parserklasse – nicht exportiert!
  const ParserClass: any = class extends CstParser {
    constructor() {
      super(allTokens);
      const $ = this as any;

      // Grammatik-Regeln binden (jede ist unten definiert)
      $.RULE("stmnt", () => rule_stmnt($, tokens));
      $.RULE("expr", () => rule_expr($, tokens));
      $.RULE("product", () => rule_product($, tokens));
      $.RULE("factor", () => rule_factor($, tokens));

      (this as any).performSelfAnalysis();
    }
  };

  // ---- einzelne Regeldefinitionen (wie def p_expr_plus in PLY) ----

  // stmnt : IDENTIFIER ASSIGN expr ';' | expr ';'
  function rule_stmnt($: any, t: any) {
    $.OR([
      {
        ALT: () => {
          $.CONSUME(t.Identifier);
          $.CONSUME(t.Assign);
          $.SUBRULE($.expr);
          $.CONSUME(t.Semicolon);
        },
      },
      {
        ALT: () => {
          $.SUBRULE2($.expr);
          $.CONSUME2(t.Semicolon);
        },
      },
    ]);
  }

  // expr : expr '+' product | expr '-' product | product
  function rule_expr($: any, t: any) {
    $.SUBRULE($.product);
    $.MANY(() => {
      $.OR([
        { ALT: () => { $.CONSUME(t.Plus); $.SUBRULE2($.product); } },
        { ALT: () => { $.CONSUME(t.Minus); $.SUBRULE3($.product); } },
      ]);
    });
  }

  // product : product '*' factor | product '/' factor | factor
  function rule_product($: any, t: any) {
    $.SUBRULE($.factor);
    $.MANY(() => {
      $.OR([
        { ALT: () => { $.CONSUME(t.Mul); $.SUBRULE2($.factor); } },
        { ALT: () => { $.CONSUME(t.Div); $.SUBRULE3($.factor); } },
      ]);
    });
  }

  // factor : '(' expr ')' | NUMBER | IDENTIFIER
  function rule_factor($: any, t: any) {
    $.OR([
      {
        ALT: () => {
          $.CONSUME(t.LParen);
          $.SUBRULE($.expr);
          $.CONSUME(t.RParen);
        },
      },
      { ALT: () => $.CONSUME(t.NumberTok) },
      { ALT: () => $.CONSUME(t.Identifier) },
    ]);
  }

  // Parser erzeugen
  return new ParserClass();
}

// Globale Parserinstanz – steht danach für Visitor & parseInput bereit
const parserInstance: any = createCalcParser(allTokens, tokens);


In [8]:
// ======================
// 4️⃣  VISITOR (Evaluator)
// ======================

// Basis-Visitor vom Parser holen
const BaseVisitor: any = (parserInstance as any).getBaseCstVisitorConstructor();

// Variablenspeicher
const Names2Values: Record<string, number> = {};

// Visitor-Klasse mit dynamischen Methoden
class CalcVisitor extends (BaseVisitor as { new (): any }) {
  constructor() {
    super();
    // TS kennt validateVisitor nicht → cast auf any
    (this as any).validateVisitor();
  }

  stmnt(ctx: any) {
    if (ctx.IDENTIFIER && ctx.ASSIGN) {
      const name = ctx.IDENTIFIER[0].image;
      const value = (this as any).visit(ctx.expr[0]);
      Names2Values[name] = value;
      return value;
    } else {
      const val = (this as any).visit(ctx.expr[0]);
      console.log(val);
      return val;
    }
  }

  expr(ctx: any) {
    let value = (this as any).visit(ctx.product[0]);
    const numProducts = ctx.product.length;
    for (let i = 1; i < numProducts; i++) {
      if (ctx.PLUS && ctx.PLUS[i - 1]) value += (this as any).visit(ctx.product[i]);
      else if (ctx.MINUS && ctx.MINUS[i - 1]) value -= (this as any).visit(ctx.product[i]);
    }
    return value;
  }

  product(ctx: any) {
    let value = (this as any).visit(ctx.factor[0]);
    const numFactors = ctx.factor.length;
    for (let i = 1; i < numFactors; i++) {
      if (ctx.MUL && ctx.MUL[i - 1]) value *= (this as any).visit(ctx.factor[i]);
      else if (ctx.DIV && ctx.DIV[i - 1]) value /= (this as any).visit(ctx.factor[i]);
    }
    return value;
  }

  factor(ctx: any) {
    if (ctx.NUMBER) {
      return parseFloat(ctx.NUMBER[0].image);
    } else if (ctx.IDENTIFIER) {
      const name = ctx.IDENTIFIER[0].image;
      if (!(name in Names2Values)) throw new Error(`Undefined variable '${name}'`);
      return Names2Values[name];
    } else {
      return (this as any).visit(ctx.expr[0]);
    }
  }
}

// Visitor-Instanz
const calcVisitor = new CalcVisitor() as any;


In [9]:
function parseInput(input: string) {
  const lexResult = CalcLexer.tokenize(input);
  parserInstance.input = lexResult.tokens;
  const cst = parserInstance.stmnt();

  if (parserInstance.errors.length > 0) {
    const err = parserInstance.errors[0];
    console.error(
      `Syntax Error near token '${err.token.image}' at line ${err.token.startLine}, column ${err.token.startColumn}.`
    );
    return;
  }

  try {
    return calcVisitor.visit(cst);
  } catch (e: any) {
    console.error(e.message);
  }
}

In [11]:
console.log("Mini-Calculator gestartet. Leerzeile beendet.\n");

while (true) {
  const s = readlineSync.question("calc> ");
  if (!s.trim()) {
    console.log("\nValues:", Names2Values);
    console.log("Beendet.");
    break;
  }
  parseInput(s);
}

Mini-Calculator gestartet. Leerzeile beendet.

25
25
25
16
9


Undefined variable 'z'



Values: { b: 16, a: 9, c: 25 }
Beendet.
